In [1]:
import os 
from dotenv import load_dotenv
from openai  import OpenAI
import gradio as gr

c:\Users\HP\Udemy\Ai\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [16]:
load_dotenv()

groq_api_key = os.getenv('GROQ_API_KEY')

groq_url = "https://api.groq.com/openai/v1"

groq = OpenAI(api_key=groq_api_key,base_url=groq_url)
MODEL = "llama-3.1-8b-instant"

## And now, writing a new callback

We now need to write a function called:

`chat(message, history)`

Which will be a callback function we will give gradio.

### The job of this function

Take a message, take the prior conversation, and return the response.

In [ ]:
system_message = "You are a helpful assistant"

In [7]:
def chat(message,history):
    return "bananas"

In [ ]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


In [ ]:
# updating callback fn
def chat(message,history):
    history = [{"role":h["role"], "content":h["content"]} for h in history] #clears out metadata
    messages = [{"role":"system","content":system_message}] + history + [{"role":"user","content":message}]
    response = groq.chat.completions.create(model = MODEL,messages = messages)
    return response.choices[0].message.content


In [20]:
view = gr.ChatInterface(fn=chat)
view.launch()

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


In [21]:
view.close()

Closing server running on port: 7865


In [26]:
#adding streaming to callback fn
def chat(message,history):
    history = [{"role":h["role"], "content":h["content"]} for h in history] #clears out metadata
    messages = [{"role":"system","content":system_message}] + history + [{"role":"user","content":message}]
    stream = groq.chat.completions.create(model = MODEL,messages = messages,stream = True)
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        yield response

In [27]:
view = gr.ChatInterface(fn=chat)
view.launch()

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


In [28]:
view.close()

Closing server running on port: 7867


# Using a system message to add context, and to give an example answer.. this is "one shot prompting" again

In [29]:
system_message = "You are a helpful assistant in a clothes store. You should try to gently encourage \
the customer to try items that are on sale. Hats are 60% off, and most other items are 50% off. \
For example, if the customer says 'I'm looking to buy a hat', \
you could reply something like, 'Wonderful - we have lots of hats - including several that are part of our sales event.'\
Encourage the customer to buy hats if they are unsure what to get."

In [30]:
view = gr.ChatInterface(fn=chat)
view.launch()

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


In [31]:
view.close()

Closing server running on port: 7867


In [32]:
system_message += "\nIf the customer asks for shoes, you should respond that shoes are not on sale today, \
but remind the customer to look at hats!"

In [33]:
view = gr.ChatInterface(fn=chat)
view.launch()

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


In [34]:
view.close()

Closing server running on port: 7867


#### Inference time technique => inserting relevant info into the prompt,to educate the model, to ans questions accurately

In [35]:
def chat(message,history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    relevant_message = system_message
    
    if "belt" in message.lower():
        relevant_message += " The store does not sell belts; if you are asked for belts, be sure to point out other items on sale."
        
    messages = [{"role":"system","content":relevant_message}] + history + [{"role":"user","content":message}]
    stream = groq.chat.completions.create(model = MODEL,messages = messages,stream = True)
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        yield response

In [36]:
view = gr.ChatInterface(fn=chat)
view.launch()

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


In [37]:
view.close()

Closing server running on port: 7867
